In [ ]:
from pathlib import Path
from typing import List, Dict, Tuple, Union
from collections import defaultdict
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm.auto import tqdm
from PIL import Image
import h5py
import scipy.io
from scipy.stats import zscore

import nilearn as nl
import nilearn.image as nl_image
import nilearn.plotting as nl_plotting
import nibabel as nib

In [ ]:
SUBJECTS = [
    f"subj{i:02d}" for i in range(1, 9)
]
data_space = "nativesurface"
beta_type ="betas_fithrf_GLMdenoise_RR"

# SUBJECTS

In [ ]:
ds_dir = '${MBS_NSD_DIR}'
# ds_path = '${MBS_NSD_DIR}/nsddata_betas/ppdata/subj01/fsaverage/betas_fithrf_GLMdenoise_RR'

ds_dir = Path(ds_dir)

# list((ds_path ).iterdir())

In [ ]:
df_stim_info= pd.read_csv(ds_dir / 'nsd_stim_info_merged.csv', index_col=0)
df_stim_info.columns

In [ ]:

nsd_expdesign = scipy.io.loadmat(ds_dir / 'nsd_expdesign.mat')

In [ ]:
nsd_expdesign.keys()

In [ ]:
# nsd_expdesign['stimpattern'].sum()
# nsd_expdesign['subjectim'].shape
# nsd_expdesign['subjectim'][1,:].max()
nsd_expdesign['masterordering'].max()

In [ ]:
df_stim_all = []

for sub in tqdm(range(1,9)):
    sub_rep_cols = [f'subject{sub}_rep{i}' for i in range(3)]
    sub_rep_mask = df_stim_info[sub_rep_cols].sum(axis=1) > 0
    df_sub = df_stim_info.loc[sub_rep_mask, ['nsdId', 'shared1000', 'flagged'] + sub_rep_cols]

    df_new = []
    for idx, row in df_sub.iterrows():
        for rep_id in range(3):
            df_new.append(
                {
                    'nsdId': row['nsdId'],
                    'trial': row[f'subject{sub}_rep{rep_id}'],
                    'rep': rep_id,
                    'shared1000': row['shared1000'],
                    'flagged': row['flagged'],
                }
            )
    df_new = pd.DataFrame(df_new)
    df_new['subject'] = sub
    df_stim_all.append(df_new)
df_stim_all = pd.concat(df_stim_all, ignore_index=True)
df_stim_all.sort_values(['subject', 'trial', 'rep'], inplace=True, ignore_index=True)
df_stim_all

In [ ]:
np.all([
    np.all((nsd_expdesign['subjectim'][sub,nsd_expdesign['masterordering']-1]-1) == df_stim_all[df_stim_all.subject == sub+1].nsdId.values)
    for sub in range(8)
])


In [ ]:
save_dir = "${MBS_DATA_PREP_OUTPUT_DIR}"
save_dir = Path(save_dir)

save_path = save_dir / f"nsd_stim_mapping.csv"

if not save_path.parent.exists():
    save_path.parent.mkdir(parents=True, exist_ok=False)
    
df_stim_all.to_csv(save_path, index=False)

In [ ]:
pd.read_csv(save_path)

In [ ]:
stim_info = defaultdict(dict)

for sub in range(1, 9):
    df_stim_subj = df_stim_all[df_stim_all.subject == sub]